In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import zipfile
import os
from datetime import datetime, timedelta
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR

In [ ]:
seed = 42
torch.random.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"

lr = 3e-4
epochs = 10
batch_sizes = {128: 32, 256: 16}

EVALUATION = True
root_path = "/personal/weather" if not EVALUATION else "/bohr/train-ma50/v2/"

# Metadata

In [ ]:
df = pd.read_csv(f"{root_path}/metadata.csv")
df.head()

In [ ]:
def pixel_to_latlon(i, j):
    """
    Converts pixel indices (row i, column j) to geographic latitude and longitude.

    Args:
        i (int): Row index (from 0 at the top to 3499 at the bottom).
        j (int): Column index (from 0 at the left to 6999 at the right).

    Returns:
        tuple: (latitude, longitude) as floats.

    This function is used to convert from image/pixel coordinates
    (such as a satellite image) to actual map coordinates.
    """
    lat = 60.0 - i * 0.01   # Each row down is 0.01 degree further south
    lon = -130.0 + j * 0.01 # Each column right is 0.01 degree further east
    return lat, lon

def solar_elevation(x, y, dt_utc):
    """
    Calculates the sun's elevation angle above the horizon for a specific
    location (pixel) and UTC time.

    Args:
        x (int): Row index in the image (vertical position).
        y (int): Column index in the image (horizontal position).
        dt_utc (datetime or str): The date and time in UTC (string or datetime).

    Returns:
        float: Solar elevation angle in degrees.

    This function tells you "how high is the sun in the sky"
    for a given place and time.
    """
    # If time is given as string, convert to datetime
    if isinstance(dt_utc, str):
        dt_utc = datetime.strptime(dt_utc, '%Y-%m-%d %H:%M:%S.%f')

    # Convert pixel indices to latitude and longitude
    lat, lon = pixel_to_latlon(x, y)

    # Estimate local time (in hours) by longitude (15 degrees = 1 hour)
    timezone_offset = lon / 15.0
    local_time = dt_utc.hour + dt_utc.minute / 60 + timezone_offset

    # Day of year (1-365/366)
    N = dt_utc.timetuple().tm_yday

    # Solar declination: angle between sun's rays and Earth's equator
    decl = 23.44 * math.sin(math.radians(360 / 365 * (N - 81)))

    # Hour angle: how far in time from solar noon
    H = 15 * (local_time - 12)  # degrees

    # Convert everything to radians for math functions
    phi = math.radians(lat)
    delta = math.radians(decl)
    H = math.radians(H)

    # Calculate elevation using spherical trigonometry
    sin_h = math.sin(phi) * math.sin(delta) + math.cos(phi) * math.cos(delta) * math.cos(H)
    h = math.degrees(math.asin(sin_h))
    return h

In [ ]:
df_train_128 = df[(df['split'] == 'train') & (df['size'] == 128)]
df_test_128 = df[(df['split'] != 'train') & (df['size'] == 128)]

df_train_256 = df[(df['split'] == 'train') & (df['size'] == 256)]
df_test_256 = df[(df['split'] != 'train') & (df['size'] == 256)]

df_train_256.head()

In [ ]:
def preprocess_metadata(df):
    start_time = pd.to_datetime(df['start_time'])

    sol, lat, lon = [], [], []
    for i in range(len(df)):
        x, y = df['i'].iloc[i], df['j'].iloc[i]

        lat_, lon_ = pixel_to_latlon(x, y)
        s = solar_elevation(x, y, start_time.iloc[i])

        lat.append(lat_)
        lon.append(lon_)
        sol.append(s)

    hours = start_time.dt.hour + start_time.dt.minute / 60
    sin_time = np.sin(2 * np.pi * hours / 24)
    cos_time = np.cos(2 * np.pi * hours / 24)

    meta = np.stack([lat, lon, sol], axis=1)
    return torch.tensor(meta, dtype=torch.float32)

meta_dim = 3

In [ ]:
metadata_train_128 = preprocess_metadata(df_train_128)
metadata_test_128 = preprocess_metadata(df_test_128)

metadata_train_256 = preprocess_metadata(df_train_256)
metadata_test_256 = preprocess_metadata(df_test_256)

metadata_test_256.shape

# Data

In [ ]:
class PatchDataset(Dataset):
    """
    Custom PyTorch dataset for image patches, segmentation masks, and optional metadata.

    Args:
        X_tensor (torch.Tensor): Input tensor of image patches, shape [N, 16, D, D]
        Y_tensor (torch.Tensor): Target segmentation masks, shape [N, D, D]
        metadata_tensor (torch.Tensor, optional): Metadata features, shape [N, D_meta]
    """
    def __init__(self, X_tensor, Y_tensor, metadata_tensor=None):
        self.X = X_tensor  # [N, 16, H, W]
        self.Y = Y_tensor  # [N, H, W]
        self.meta = metadata_tensor  # [N, D_meta] or None

    def __len__(self):
        return len(self.X) // 1

    def __getitem__(self, idx):
        x = self.X[idx].to(torch.float32)
        y = self.Y[idx].unsqueeze(0).to(torch.float32)  # [1, H, W]
        meta = self.meta[idx].to(torch.float32)  # [D_meta]
        return x, y, meta

In [ ]:
def prepare_dataset(data, threshold=0.1):
    """
    Prepares input (X) and output (Y) tensors from raw patch data,
    applying normalization, thresholding, and cleaning.

    Args:
        data (dict): Dictionary mapping patch size to lists of numpy arrays,
                     each array is [17, D, D] (16 channels + 1 mask channel).
        threshold (float): Threshold for converting last channel to binary mask.

    Returns:
        tuple: (X_dict, Y_dict, norm_stats)
            - X_dict: Dict of input tensors, shape [N, 16, D, D] for each size.
            - Y_dict: Dict of output masks, shape [N, D, D] for each size.
            - norm_stats: Dict with 'mean' and 'std' for normalization (per channel).

    This function does two passes over the data:
        1. Computes mean and std for each input channel (ignoring NaNs).
        2. Normalizes the data and packs it into torch tensors.
    """

    num_channels = 16  # First 16 channels are features, last one is target mask

    # === Pass 1: Compute statistics for normalization ===
    sum_channels = torch.zeros(num_channels, dtype=torch.float64)    # Total sum per channel
    sum_sq_channels = torch.zeros(num_channels, dtype=torch.float64) # Total squared sum per channel
    count_channels = torch.zeros(num_channels, dtype=torch.int64)    # Number of valid (non-NaN) values per channel

    for patches in data.values():
        for arr in patches:
            arr = arr.squeeze(0).detach().clone()  # arr: [17, D, D]
            x = arr[:16]  # [16, D, D] - input features
            channel_mean = torch.nanmean(x, dim=(1,2), keepdim=True)
            x = torch.where(torch.isnan(x), channel_mean, x)

            valid = ~torch.isnan(x)  # Mask of valid values
            sum_channels += torch.where(valid, x, torch.tensor(0.0)).sum(dim=(1, 2))
            sum_sq_channels += torch.where(valid, x ** 2, torch.tensor(0.0)).sum(dim=(1, 2))
            count_channels += valid.sum(dim=(1, 2))

    # Compute mean and std per channel
    means = sum_channels / count_channels
    variances = (sum_sq_channels / count_channels) - means ** 2
    stds = torch.sqrt(torch.clamp(variances, min=1e-6))  # avoid sqrt of negative

    # === Make sure no zeros, NaNs or infs in std/mean ===
    stds[stds == 0] = 1.0
    stds[torch.isnan(stds)] = 1.0
    stds[torch.isinf(stds)] = 1.0

    means[torch.isnan(means)] = 0.0
    means[torch.isinf(means)] = 0.0

    norm_stats = {
        "mean": means.to(torch.float32),
        "std": stds.to(torch.float32)
    }

    # === Pass 2: Normalize and pack tensors for PyTorch training ===
    X_dict = {}
    Y_dict = {}

    for size, patches in data.items():
        n = len(patches)    # Number of patches for this size
        D = size            # Patch size (width and height)

        # Allocate memory for all normalized patches and masks
        X_tensor = torch.empty((n, num_channels, D, D), dtype=torch.float16)
        Y_tensor = torch.empty((n, D, D), dtype=torch.uint8)

        for i, arr in enumerate(patches):
            arr = arr.squeeze(0).detach().clone()  # [17, D, D]
            x = arr[:16]           # First 16 channels: features
            y = (arr[16] > threshold).to(torch.uint8)  # Last channel: mask, binarized

            channel_mean = torch.nanmean(x, dim=(1,2), keepdim=True)
            x = torch.where(torch.isnan(x), channel_mean, x)

            # Normalize per channel: (value - mean) / std
            x_norm = ((x - means[:, None, None]) / stds[:, None, None]).to(torch.float16)

            X_tensor[i] = x_norm   # Save normalized input
            Y_tensor[i] = y        # Save output mask

        X_dict[size] = X_tensor
        Y_dict[size] = Y_tensor

    return X_dict, Y_dict, norm_stats

def build_dataloaders(X_dict, Y_dict, metadata, shuffle=True):
    """
    Builds PyTorch DataLoader objects for different patch sizes.

    Args:
        X_dict (dict): Dict of input tensors for each patch size.
        Y_dict (dict): Dict of label tensors for each patch size.
        shuffle (bool): Whether to shuffle the data (good for training).
        regime (str): Can be 'train' or 'test', not used in code.

    Returns:
        dict: Dictionary of DataLoader objects for each patch size.
    """
    train_loaders = {}
    for size in X_dict:
        ds = PatchDataset(X_dict[size], Y_dict[size], metadata[size])  # Make dataset for this size
        train_loaders[size] = DataLoader(ds, batch_size=batch_sizes[size], shuffle=shuffle)
    return train_loaders

In [ ]:
# Load from .npz file
loaded = np.load(f"{root_path}/dataset.npz")

# Reconstruct dictionaries
X_train_128 = torch.from_numpy(loaded['X_train_128']) # [529, 16, 128, 128]
y_train_128 = torch.from_numpy(loaded['Y_train_128']).unsqueeze(1) # [529, 1, 128, 128]
train_128 = torch.cat([X_train_128, y_train_128], dim=1) # [529, 17, 128, 128]

X_train_256 = torch.from_numpy(loaded['X_train_256'])
y_train_256 = torch.from_numpy(loaded['Y_train_256']).unsqueeze(1)
train_256 = torch.cat([X_train_256, y_train_256], dim=1)

train_dataset = {
    128: train_128,
    256: train_256
}

X_test_128 = torch.from_numpy(loaded['X_test_128'])
y_test_128 = torch.from_numpy(loaded['Y_test_128']).unsqueeze(1)
test_128 = torch.cat([X_test_128, y_test_128], dim=1)

X_test_256 = torch.from_numpy(loaded['X_test_256'])
y_test_256 = torch.from_numpy(loaded['Y_test_256']).unsqueeze(1)
test_256 = torch.cat([X_test_256, y_test_256], dim=1)

test_dataset = {
    128: test_128,
    256: test_256
}

del loaded

In [ ]:
X_train, Y_train, train_norm = prepare_dataset(train_dataset)
X_test, Y_test, test_norm = prepare_dataset(test_dataset)

In [ ]:
X_train[128].shape

In [ ]:
for dx, dname in zip([X_train, Y_train],
                     ['X_train', 'Y_train']
                    ):
    for k, d in dx.items():
        print(dname, k, d.shape)

In [ ]:
metadata_train = {
    128: metadata_train_128,
    256: metadata_train_256,
}

metadata_test = {
    128: metadata_test_128,
    256: metadata_test_256,
}

In [ ]:
train_loaders = build_dataloaders(X_train, Y_train, metadata_train, shuffle=True)
test_loaders = build_dataloaders(X_test, Y_test, metadata_test, shuffle=False)

In [ ]:
# sanity check
batch = next(iter(train_loaders[128]))
[b.shape for b in batch]

# Model

In [ ]:
class ASPP(nn.Module):
    # Atrous Spatial Pyramid Pooling
    def __init__(self, in_ch, out_ch):
        super().__init__()
        rates = [1,6,12,18]
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(8, out_ch),
                nn.ReLU(inplace=True)
            ) for r in rates
        ])
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch,1,bias=False),
            nn.GroupNorm(8,out_ch),
            nn.ReLU(inplace=True)
        )
        self.project = nn.Sequential(
            nn.Conv2d(len(rates)*out_ch + out_ch, out_ch,1,bias=False),
            nn.GroupNorm(8,out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
    def forward(self, x):
        res = [c(x) for c in self.convs]
        gp  = F.interpolate(self.global_pool(x),
                            size=x.shape[2:], mode='bilinear',
                            align_corners=False)
        res.append(gp)
        x = torch.cat(res, dim=1)
        return self.project(x)

class SEBlock(nn.Module):
    # Squeeze-and-Excitation Networks
    def __init__(self, channels, r=16):
        super().__init__()
        self.fc1 = nn.Conv2d(channels, channels//r, 1)
        self.fc2 = nn.Conv2d(channels//r, channels, 1)
    def forward(self, x):
        s = x.mean((2,3), keepdim=True)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s

class UNet(nn.Module):
    def __init__(self, in_channels: int, meta_dim: int):
        super().__init__()

        self.channel_scales = nn.Parameter(torch.ones(1, in_channels, 1, 1))
        self.lambda_l1 = 1e-4

        self.squeeze_blk = SEBlock(channels=16)

        self.encoder1 = self.conv_block(in_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.mid = ASPP(512, 1024)

        self.meta_mlp = nn.Sequential(
            nn.Linear(meta_dim, 256),
            nn.LeakyReLU(0.02),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.02),
            nn.Linear(512, 1024),
        )

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self.conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)

        self.final = nn.Conv2d(64, 1, kernel_size=1)  # Output: logits per pixel

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, metadata):
        B = x.size(0)

        x = self.squeeze_blk(x)
        x = x * self.channel_scales

        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))

        # Bottleneck
        m = self.mid(self.pool(e4)) # [B, 1024, H/16, W/16]

        meta_feats = self.meta_mlp(metadata) # [B, 1024]
        meta_feats = meta_feats.view(B, 1024, 1, 1)

        m = m + meta_feats

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(m), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.final(d1)  # Output shape: [batch, 1, H, W]

    def get_l1_penalty(self):
        return self.lambda_l1 * torch.sum(torch.abs(self.channel_scales))

In [ ]:
model = UNet(in_channels=16, meta_dim=meta_dim).to(device)
model.load_state_dict(torch.load(f"{root_path}/model_weights.pth"), strict=False)

x, m = batch[0].to(device), batch[2].to(device)
model(x, m).shape

# Training

In [ ]:
def dice_loss(pred, target, eps=1e-6):
    # Apply sigmoid to logits to get probabilities between 0 and 1
    pred = torch.sigmoid(pred)
    # Intersection and union for each image in the batch
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    # Dice loss = 1 - Dice coefficient (higher is worse)
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()

def get_loss(logits, target):
    dice = dice_loss(logits, target)
    return dice + model.get_l1_penalty()

In [ ]:
epochs = 20
lr = 1e-4

In [ ]:
losses, lrs = [], []

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=lr,
             steps_per_epoch=(len(train_loaders[128]) + len(train_loaders[256])), epochs=epochs,
             pct_start=0.1, div_factor=100, final_div_factor=100)
scaler = GradScaler(device)

In [ ]:
def evaluate_on_test(model, test_loaders, device="cuda", threshold=0.5, do_print=True):
    """
    Evaluates the segmentation model on test data and prints metrics.

    Args:
        model (nn.Module): The trained segmentation model.
        test_loaders (dict): Dictionary of DataLoader(s) for each patch size.
        device (str): 'cuda' for GPU or 'cpu' for CPU.
        threshold (float): Threshold for converting probabilities to binary masks.

    Prints:
        - IoU (Intersection over Union)
        - Dice coefficient
        - Precision
        - Recall
        - Image-level accuracy (if any pixel is detected as "rain" in image)
    Returns:
        float: Average of Dice coefficient and image-level accuracy.
    """

    model.eval()             # Set model to evaluation mode (no dropout/batchnorm update)
    model.to(device)

    total_iou = 0.0          # Intersection over Union accumulator
    total_dice = 0.0         # Dice coefficient accumulator
    total_prec = 0.0         # Precision accumulator
    total_recall = 0.0       # Recall accumulator
    total_batches = 0

    rain_y_true = []         # List for true image-level labels (rain/no rain)
    rain_y_pred = []         # List for predicted image-level labels

    with torch.no_grad():    # No need to compute gradients during evaluation
        for size, loader in test_loaders.items():
            pbar = tqdm(loader, desc=f"Test {size}x{size}", leave=False)
            for x, y, m in pbar:
                x = x.to(device)
                y = y.to(device)
                m = m.to(device)

                logits = model(x, m)  # [B, 1, H, W]
                probs = torch.sigmoid(logits)        # Probabilities in [0, 1]
                preds = (probs > threshold).float()  # Binary predictions

                # Compute intersection and union for IoU
                intersection = (preds * y).sum(dim=(1, 2, 3))
                union = ((preds + y) > 0).float().sum(dim=(1, 2, 3))
                iou = (intersection / (union + 1e-6)).mean().item()

                # Compute Dice coefficient
                dice = (2 * intersection / (preds.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) + 1e-6)).mean().item()

                # Precision and recall for all pixels
                tp = (preds * y).sum().item()                 # True positives
                fp = (preds * (1 - y)).sum().item()           # False positives
                fn = ((1 - preds) * y).sum().item()           # False negatives

                precision = tp / (tp + fp + 1e-6)
                recall = tp / (tp + fn + 1e-6)

                # Add to totals for averaging
                total_iou += iou
                total_dice += dice
                total_prec += precision
                total_recall += recall
                total_batches += 1

                # For image-level rain/no-rain: True if any pixel is "rain"
                rain_y_true += [(y > 0.5).any(dim=(1,2,3)).cpu()]
                rain_y_pred += [(preds > 0.5).any(dim=(1,2,3)).cpu()]

    if total_batches == 0:
        print("⚠️ No valid batches")
        return

    # Combine image-level labels for accuracy calculation
    rain_y_true = torch.cat(rain_y_true)
    rain_y_pred = torch.cat(rain_y_pred)
    acc = (rain_y_true == rain_y_pred).float().mean().item()

    dice_final = total_dice / total_batches
    total_score = (dice_final + acc) / 2

    if do_print:
        print(f"\nTHRESHOLD = {threshold}")
        # print(f"Test metrics across all sizes:")
        # print(f"IoU   : {total_iou / total_batches:.4f}")
        # print(f"Dice  : {dice_final:.4f}")
        # print(f"Prec  : {total_prec / total_batches:.4f}")
        # print(f"Recall: {total_recall / total_batches:.4f}")
        # print(f"Image-level Rain Acc: {acc:.4f}")
        print(f"Final score: {total_score:.4f}")

    # Return a combined score for leaderboard
    return dice_final, acc, total_score

In [ ]:
for epoch in range(1, epochs+1):
    model.train()
    running_loss = 0.0

    for sz in train_loaders:
        for images, masks, metadata in tqdm(train_loaders[sz]):
            images, masks, metadata = images.to(device), masks.to(device), metadata.to(device)

            # forward pass
            with autocast(device):
                logits = model(images, metadata)
                logits_clamped = torch.clamp(logits, -10, 10) # for stability
                loss = get_loss(logits, masks)

            # backward pass
            if torch.isnan(loss):
                print("NAN LOSS")
                continue

            optimizer.zero_grad()
            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            # stats
            running_loss += loss.item()
            losses.append(loss.item())
            lrs.append(optimizer.param_groups[0]["lr"])

    avg_loss = running_loss / (len(train_loaders[128]) + len(train_loaders[256]))
    dice_final, acc, total_score = evaluate_on_test(model, test_loaders, threshold=0.5, do_print=False)

    if total_score > best_score:
        best_score = total_score
        torch.save(model.state_dict(), "unet.pth")
        print("Saving new best model...")

    print(f"Epoch {epoch}; loss={avg_loss:.4f}; score={total_score:.4f} (dice={dice_final:.2f}, acc={acc:.2f})")

In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(losses)

In [ ]:
len(losses)

In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(lrs)

In [ ]:
plt.figure(figsize=(12, 3))
l = torch.tensor(losses[:1200]).reshape(-1, 100).mean(dim=1).detach().cpu()
plt.plot(l)

# Evaluation

In [ ]:
# best_threshold = find_best_thresh(model, test_loaders)
totals, thresholds = [], []
best_thresh, best_total = 0, 0

for threshold in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    _, _, total = evaluate_on_test(model, test_loaders, threshold=threshold)
    totals.append(total)
    thresholds.append(threshold)

    if total > best_total:
        best_total = total
        best_thresh = threshold

In [ ]:
plt.plot(thresholds, totals)

In [ ]:
best_thresh, best_total

# Submission

In [ ]:
Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i].unsqueeze(0).to(torch.float32).to(device)
        m = metadata_test[128][i].unsqueeze(0).to(torch.float32).to(device)

        logits = model(x, m)

        probs = torch.sigmoid(logits)
        preds = (probs > best_thresh).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i].unsqueeze(0).to(torch.float32).to(device)
        m = metadata_test[256][i].unsqueeze(0).to(torch.float32).to(device)

        logits = model(x, m)

        probs = torch.sigmoid(logits)
        preds = (probs > best_thresh).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# You must name your prediction arrays `Y_pred_128` and `Y_pred_256`, and name the file `pred_a.npz` for the public leaderboard
np.savez('pred_a.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)

In [ ]:
if not EVALUATION:
    1/0

In [ ]:
# Your notebook will gain access to the test dataset via the DATA_PATH environment variable after submission.
# In the actual debugging process, it is normal for this block to fail to run.
#‘DATA_PATH’ is an environment variable provided by the testing machine, used to read the testing set.
# Participants cannot access it directly. Please retain this environment variable during submission, otherwise the testing machine will not be able to read the testing set.
TEST_PATH = os.environ.get('DATA_PATH', "test")

test = np.load(Path(TEST_PATH) / "X_test.npz") # Read test data from X_test.npz under the provided path

X_test = {
    128: torch.from_numpy(test['X_test_128']),
    256: torch.from_numpy(test['X_test_256']),
} # only X_test will be provided

df_test = pd.read_csv(Path(TEST_PATH) / "metadata_test.csv") # Read metadata from metadata_test.csv under the provided path

metadata_test_128 = preprocess_metadata(df_test[df_test["size"] == 128])
metadata_test_256 = preprocess_metadata(df_test[df_test["size"] == 256])
metadata = {
    128: metadata_test_128,
    256: metadata_test_256
}

In [ ]:
Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i].unsqueeze(0).to(torch.float32).to(device)
        m = metadata_test[128][i].unsqueeze(0).to(torch.float32).to(device)

        logits = model(x, m)

        probs = torch.sigmoid(logits)
        preds = (probs > best_thresh).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i].unsqueeze(0).to(torch.float32).to(device)
        m = metadata_test[256][i].unsqueeze(0).to(torch.float32).to(device)

        logits = model(x, m)

        probs = torch.sigmoid(logits)
        preds = (probs > best_thresh).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# You must name your prediction arrays `Y_pred_128` and `Y_pred_256`, and name the file `pred_b.npz` for private leaderboard
np.savez('pred_b.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)

In [ ]:
# zip `pred_a.npz` and `pred_b.npz` into `submission.zip`
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('pred_a.npz')
    zipf.write('pred_b.npz')